In [1]:
# =============================================================================
# 🔨 06_TheForge.ipynb — Cell 1: 환경 + 팩터 레지스트리 + 레짐별 매핑
# =============================================================================
#
# [목적]
# 멀티팩터 조합 모델 구축
# 05_Factor v4 재검증 결과를 기반으로 레짐별 팩터 조합 설계
#
# [구조]
# Cell 1: 환경 + 팩터 레지스트리 + 레짐 매핑 (← 여기)
# Cell 2: 전 팩터 시그널 로드 + 통합 패널
# Cell 3: 팩터 간 상관 매트릭스 (미확인 쌍 포함)
# Cell 4: Phase 2 — 레짐별 동일가중 조합 백테스트 (베이스라인)
# Cell 5: Phase 3 — Bear 자산배분 레이어 (현금/채권 비중)
# Cell 6: Phase 4 — 이벤트 트리거 (G-1, T-1) 오버레이
# Cell 7: Phase 5 — 거래비용 민감도 + Walk-Forward
# Cell 8: 최종 결론 주석
#
# [핵심 설계 원칙]
# 1. H는 always-on 베이스 팩터 (전 레짐 WIN, t=12.3/7.7)
# 2. 레짐 스위치에 따라 보조 팩터 on/off
# 3. Bear 구간: 팩터 틸트(종목선정) + 자산배분(현금/채권) 분리
# 4. 이벤트 트리거(G-1, T-1)는 Core 스코어 가감 방식
# 5. 처음은 동일가중 → 과최적화 금지
#
# [레짐 체계]
# regime_v4 (3레짐 + bear_phase 보조):
#   Bull:    S&P500 > 10M SMA (상승장)
#   Bear:    S&P500 < 10M SMA (하락장)
#   Neutral: Bear→Bull 전환 후 3개월 이내
#   bear_phase: declining(MA 하향) / recovering(MA 상향)
#
# [v2→v4 레짐 매핑 참고]
#   v2 Expansion, Peak, Recovery_Late → v4 Bull
#   v2 Contraction, Crash             → v4 Bear
#   v2 Neutral, Recovery_Early        → v4 Neutral (정확히 1:1은 아님, 주의)
#   ⚠ v4에서는 v2의 세분화를 포기하고 단순화한 대신
#      bear_phase로 Bear 내부를 구분
#
# [백테스트 기간]
# 2013-06 ~ 2026-02 (약 151개월)
# 유니버스: 503 tickers (S&P500 + 과거 구성종목)
# 벤치마크: EW (동일가중)
# 거래비용: 20bp 편도 기준
# 리밸런싱: 월말 (이벤트 트리거 제외)
# Top N: 30 (기본)
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ─── 경로 (SSOT) ─────────────────────────────────────────────────────────────
QP2_ROOT = Path(r"C:\QP2")
DATA_DIR = QP2_ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
INTERIM  = DATA_DIR / "interim"
META_DIR = DATA_DIR / "meta"

PATHS = dict(
    px_wide     = INTERIM / "yahoo_adjclose_wide.parquet",
    ret_1m      = INTERIM / "ret_1m_wide.parquet",       # 월간 수익률 wide
    fund_q      = INTERIM / "fundamentals_quarterly.parquet",
    mktcap_m    = INTERIM / "mktcap_monthly.parquet",
    regime_v4   = INTERIM / "regime_v4.parquet",          # ★ v4 레짐
    regime_v2   = INTERIM / "regime_indicators_combined.parquet",  # 참고용
    universe    = META_DIR / "sp500_universe.parquet",
    fscore      = INTERIM / "fscore_signal.parquet",
    nsi         = INTERIM / "nsi_annual.parquet",
    insider     = INTERIM / "insider_mspr.parquet",
    p7_panel    = INTERIM / "p7_signal_panel.parquet",
)

# ─── 글로벌 파라미터 ──────────────────────────────────────────────────────────
TOP_N       = 30        # 포트폴리오 종목 수
COST_BP     = 20        # 편도 거래비용 (bp)
REBAL_FREQ  = "ME"      # 월말 리밸런싱
BT_START    = "2013-06"
BT_END      = "2026-02"

# ─── 유틸리티 함수 ────────────────────────────────────────────────────────────
def winsorize(s, lower=0.01, upper=0.99):
    """시리즈 윈저라이즈 (cross-sectional에서 날짜별로 사용)"""
    lo, hi = s.quantile(lower), s.quantile(upper)
    return s.clip(lo, hi)

def zscore_by_date(df, col, group_col="date"):
    """날짜별 횡단면 z-score"""
    return df.groupby(group_col)[col].transform(
        lambda x: (x - x.mean()) / x.std()
    )

def calc_perf(cum_ret):
    """누적수익률 시리즈 → CAGR, Sharpe, MaxDD"""
    total = cum_ret.iloc[-1] / cum_ret.iloc[0]
    years = len(cum_ret) / 12
    cagr  = total ** (1/years) - 1
    monthly_ret = cum_ret.pct_change().dropna()
    sharpe = monthly_ret.mean() / monthly_ret.std() * np.sqrt(12) if monthly_ret.std() > 0 else 0
    dd = cum_ret / cum_ret.cummax() - 1
    maxdd = dd.min()
    return {"CAGR": cagr, "Sharpe": sharpe, "MaxDD": maxdd}

def calc_tstat(excess_monthly):
    """월간 초과수익률 → t-stat"""
    n = len(excess_monthly)
    if n < 3:
        return 0.0
    return excess_monthly.mean() / (excess_monthly.std() / np.sqrt(n))

def backtest_topn(signal_wide, ret_wide, n=TOP_N, cost_bp=COST_BP):
    """
    Top-N EW 포트폴리오 백테스트
    signal_wide: date × ticker (z-score, 높을수록 좋음)
    ret_wide:    date × ticker (월간 수익률)
    Returns: DataFrame with port_ret, bm_ret, excess, cum_port, cum_bm
    """
    common_dates = signal_wide.index.intersection(ret_wide.index)
    common_dates = sorted(common_dates)
    
    results = []
    prev_holdings = set()
    
    for i, dt in enumerate(common_dates[:-1]):
        next_dt = common_dates[i + 1]
        
        # 시그널 기준 Top N 선정
        scores = signal_wide.loc[dt].dropna()
        if len(scores) < n:
            continue
        top_n = scores.nlargest(n).index.tolist()
        
        # 수익률
        rets = ret_wide.loc[next_dt, top_n].dropna()
        if len(rets) == 0:
            continue
        port_ret = rets.mean()
        bm_ret   = ret_wide.loc[next_dt].dropna().mean()
        
        # 거래비용 (턴오버 기반)
        curr_set = set(top_n)
        if prev_holdings:
            turnover = len(curr_set - prev_holdings) / n
        else:
            turnover = 1.0
        cost = turnover * cost_bp * 2 / 10000  # 편도 × 2 (매수+매도)
        
        results.append({
            "date": next_dt,
            "port_ret_gross": port_ret,
            "port_ret_net": port_ret - cost,
            "bm_ret": bm_ret,
            "turnover": turnover,
            "n_stocks": len(rets),
        })
        prev_holdings = curr_set
    
    df = pd.DataFrame(results).set_index("date")
    df["excess_gross"] = df["port_ret_gross"] - df["bm_ret"]
    df["excess_net"]   = df["port_ret_net"] - df["bm_ret"]
    df["cum_port"]     = (1 + df["port_ret_net"]).cumprod()
    df["cum_bm"]       = (1 + df["bm_ret"]).cumprod()
    return df


# =============================================================================
# ★ 팩터 레지스트리 — v4 재검증 최종 결과 기반
# =============================================================================
#
# 각 팩터의 메타데이터를 딕셔너리로 정의.
# 멀티팩터 조합 시 이 레지스트리를 참조하여 on/off, 가중치 등을 결정.
#
# 필드 설명:
#   name:        팩터 ID
#   desc:        한줄 설명
#   type:        "score" (z-score 기반, 합산 가능) / "filter" (이진, 종목 제외용)
#                / "event" (이벤트 트리거, 비정기)
#   direction:   "long" (높을수록 좋음) / "short" (낮을수록 좋음)
#   regimes:     유효 레짐 리스트 ["Bull", "Bear", "Neutral"]
#   role:        "main" / "aux" / "filter" / "avoid"
#   rebal:       "monthly" / "event"
#   tstat:       v4 재검증 대표 t-stat (레짐별 최고값)
#   params:      주요 파라미터
#   signal_src:  시그널 생성 소스 (어디서 로드/계산하는지)
#   note:        특이사항
#
# ⚠ 이 레지스트리는 "설계 사양서"이지 코드가 아님.
#    Cell 2에서 실제 시그널을 로드하고,
#    Cell 4에서 이 매핑대로 조합한다.
# =============================================================================

FACTOR_REGISTRY = {
    # ── always-on ──────────────────────────────────────────────────────────
    "H": {
        "desc":      "섹터 모멘텀 (3M lookback)",
        "type":      "score",
        "direction": "long",
        "regimes":   ["Bull", "Bear", "Neutral"],   # 전 레짐 WIN
        "role":      "main",
        "rebal":     "monthly",
        "tstat":     {"Bull": 12.3, "Bear": 7.7, "Neutral": 2.16},
        "params":    {"lookback_m": 3},
        "signal_src": "02_H.ipynb → h_z_wide (date×ticker z-score)",
        "note":      "MVP. IT 편향 주의. 전 레짐 베이스로 항상 ON.",
    },
    
    # ── Bull 팩터 ──────────────────────────────────────────────────────────
    "T-1": {
        "desc":      "리더 급등 spillover (Bull 전용 가산)",
        "type":      "event",
        "direction": "long",
        "regimes":   ["Bull"],
        "role":      "aux",
        "rebal":     "event",   # hold=7d, 비정기
        "tstat":     {"Bull": 3.21},
        "params":    {"surge_sigma": 2.5, "hold_d": 7, "tier": "Top_Q4"},
        "signal_src": "02_T-1.ipynb → 인라인 계산 (일간 이벤트)",
        "note":      "Core 스코어에 z-score 가산. 리밸런싱 시 직전 7일 이벤트 체크.",
    },
    "P-7": {
        "desc":      "Net Stock Issuance (자사주 매입 = 높은 점수)",
        "type":      "score",
        "direction": "long",
        "regimes":   ["Bull"],
        "role":      "aux",
        "rebal":     "monthly",
        "tstat":     {"Bull": 2.43},
        "params":    {"n_top": 50},   # N=50에서 t=2.20
        "signal_src": "02_P-7.ipynb → nsi_score (nsi_annual.parquet)",
        "note":      "회전율 7.4%로 공짜에 가까움. Crash에서 역작동 주의.",
    },
    "G-1_bull": {
        "desc":      "급락 반전 매수 (Bull, lookback=3M)",
        "type":      "event",
        "direction": "long",
        "regimes":   ["Bull"],
        "role":      "aux",
        "rebal":     "monthly",   # hold=20 → 사실상 월간
        "tstat":     {"Bull": 2.55},
        "params":    {"lookback_m": 3, "hold_d": 20},
        "signal_src": "05_G-1.ipynb → 인라인 계산",
        "note":      "hold=20이라 월간 리밸런싱에 편입 가능.",
    },
    
    # ── Bear 팩터 ──────────────────────────────────────────────────────────
    "A-3": {
        "desc":      "Value × Catalyst (저평가+개선 기업)",
        "type":      "score",
        "direction": "long",
        "regimes":   ["Bear"],
        "role":      "main",
        "rebal":     "monthly",
        "tstat":     {"Bear": 2.21},
        "params":    {"lag_days": 45, "value_filter_pct": 0.5},
        "signal_src": "02_A.ipynb → signal_df (value_filter + catalyst_z)",
        "note":      "큰 알파(+72.6%) but 큰 변동. Bear 메인이지만 현금/채권이 우월함을 잊지 말 것.",
    },
    "G-1_bear": {
        "desc":      "급락 반전 매수 (Bear, lookback=5M)",
        "type":      "event",
        "direction": "long",
        "regimes":   ["Bear"],
        "role":      "aux",
        "rebal":     "monthly",
        "tstat":     {"Bear": 7.78},
        "params":    {"lookback_m": 5, "hold_d": 20},
        "signal_src": "05_G-1.ipynb → 인라인 계산",
        "note":      "v4 재검증 최고 t-stat. Bear 급락 반전 시 강력.",
    },
    "P-5": {
        "desc":      "저베타 (Betting Against Beta)",
        "type":      "score",
        "direction": "long",   # 저베타 = 높은 BAB score
        "regimes":   ["Bear"],
        "role":      "stability",
        "rebal":     "monthly",
        "tstat":     {"Bear": 4.25},
        "params":    {"window_m": 12},
        "signal_src": "05_E_P5.ipynb → beta_long → -beta z-score",
        "note":      "v4 부활. Bear CAGR=-18.21%로 '덜 잃기'. 회전율 10.8%.",
    },
    "E-5": {
        "desc":      "저변동 (Low Idiosyncratic Vol)",
        "type":      "score",
        "direction": "long",   # 저변동 = 높은 점수
        "regimes":   ["Bear"],
        "role":      "stability",
        "rebal":     "monthly",
        "tstat":     {"Bear": 4.70},
        "params":    {"window_m": 6},
        "signal_src": "05_E_P5.ipynb → ivol → -ivol z-score",
        "note":      "v4 부활. P-5와 상관 -0.37 (반대 방향 → 분산 효과).",
    },
    "D-1": {
        "desc":      "단순 모멘텀 12-1 (Bear 보조)",
        "type":      "score",
        "direction": "long",
        "regimes":   ["Bear"],
        "role":      "aux",
        "rebal":     "monthly",
        "tstat":     {"Bear": 0.18},
        "params":    {"lookback": "MOM_12_1"},
        "signal_src": "02_D.ipynb → mom_signals['MOM_12_1']",
        "note":      "방향 맞지만 불안정 (t=0.18). 보조로만.",
    },
    "G-1b": {
        "desc":      "급등 회피 시그널 (Bear/Neutral)",
        "type":      "filter",
        "direction": "short",  # 급등 종목 회피 → 낮을수록 좋음(=감점)
        "regimes":   ["Bear", "Neutral"],
        "role":      "avoid",
        "rebal":     "monthly",
        "tstat":     {"Bear": -2.05, "Neutral": -2.31},
        "params":    {},
        "signal_src": "05_G-1.ipynb → 급등 종목 플래그",
        "note":      "Bear에서 급등=함정. 합산 스코어에서 감점 처리.",
    },
    
    # ── Neutral 팩터 ───────────────────────────────────────────────────────
    "D-3": {
        "desc":      "변동성 조정 모멘텀 (MOM_3_1 / VOL)",
        "type":      "score",
        "direction": "long",
        "regimes":   ["Neutral"],
        "role":      "main",
        "rebal":     "monthly",
        "tstat":     {"Neutral": 2.52},   # Sharpe 기준
        "params":    {"lookback": "MOM_3_1", "vol_window": "6M"},
        "signal_src": "02_D.ipynb → mom_vol_adj (MOM_3_1/VOL)",
        "note":      "v4에서 MOM_3_1/VOL로 재정의. Neutral 메인.",
    },
    
    # ── 리스크 필터 (단독 팩터 아님) ──────────────────────────────────────
    "F-1": {
        "desc":      "Piotroski F-score (리스크 필터)",
        "type":      "filter",
        "direction": "long",  # 높은 F-score = 건전
        "regimes":   ["Bear"],
        "role":      "filter",
        "rebal":     "monthly",
        "tstat":     {"Bear": 0.09},   # v4에서 대폭 약화
        "params":    {"version": "C", "lag_days": 0},
        "signal_src": "02_F.ipynb → fscore_signal.parquet",
        "note":      "v2 +11%p → v4 +0.4%. A-3 필터(저품질 제거)로 전환 고려.",
    },
}


# =============================================================================
# ★ 레짐별 팩터 매핑 — 06_Multi 설계 핵심
# =============================================================================
#
# 각 레짐에서 어떤 팩터를 어떤 역할로 사용할지 정의.
# Phase 2에서는 동일가중으로 시작 (weight는 나중에 최적화).
#
# 구조: {레짐: [팩터ID 리스트]}
# "score" 타입만 z-score 합산 대상.
# "filter" 타입은 합산 후 하위 종목 제외.
# "event" 타입은 합산 후 가산/감점.
# =============================================================================

REGIME_FACTOR_MAP = {
    "Bull": {
        "scores":  ["H", "G-1_bull", "P-7"],    # z-score 합산 (동일가중)
        "events":  ["T-1"],                       # 이벤트 시 가산
        "filters": [],                            # Bull에서는 필터 없음
        "avoids":  [],                            # Bull에서 급등 = 모멘텀 (회피 안 함)
    },
    "Bear": {
        "scores":  ["H", "A-3", "P-5", "E-5", "D-1"],  # z-score 합산
        "events":  ["G-1_bear"],                         # 급락 반전 시 가산
        "filters": ["F-1"],                              # 저품질 제거
        "avoids":  ["G-1b"],                             # 급등 종목 감점
    },
    "Neutral": {
        "scores":  ["H", "D-3"],                 # z-score 합산
        "events":  [],                            # Neutral은 이벤트 팩터 없음
        "filters": [],
        "avoids":  ["G-1b"],                      # 급등 종목 감점
    },
}

# =============================================================================
# ★ Bear 자산배분 파라미터 — Phase 3
# =============================================================================
#
# Bear에서 현금/채권 비중 조절 로직.
# 이진법(0/100)이 아니라 점진적 전환.
#
# Bear 진입:
#   1개월차 → cash_weight = 0.30
#   2개월차 → cash_weight = 0.50
#   3개월+ → cash_weight = 0.70 (최대)
#
# Bear 탈출 (bear_phase: declining → recovering):
#   recovering 1개월차 → cash_weight 한 단계 축소
#   Bull 확정(S&P > 10M SMA) → 한 단계 더 축소
#   2~3개월에 걸쳐 완전 복귀
#
# ⚠ Bear 팩터의 진짜 역할: 덜 잃기. 현금/채권(0%)이 모든 주식 전략보다 우월.
#    팩터 틸트는 "레짐 전환 초기" + "자산배분 결정 전 버퍼" 용도.
# =============================================================================

BEAR_ALLOC = {
    "bear_month_1": 0.30,   # 주식 70%
    "bear_month_2": 0.50,   # 주식 50%
    "bear_month_3+": 0.70,  # 주식 30% (최대 현금)
    "recovering_step": -0.20,  # recovering 전환 시 현금 비중 20%p 감소
    "bull_confirmed_step": -0.20,  # Bull 확정 시 추가 20%p 감소
    "min_cash": 0.0,        # 최소 현금 (Bull 확정 후)
    "max_cash": 0.70,       # 최대 현금
}

# =============================================================================
# ★ 이벤트 트리거 파라미터 — Phase 4
# =============================================================================
#
# 정기 리밸런싱(월말): H, A-3, D-3, P-5, E-5, P-7, D-1
# 이벤트 트리거: G-1(급락), T-1(리더 급등)
#
# 설계:
#   Core 스코어 = Σ(레짐별 score 팩터 z-scores) (동일가중)
#   이벤트 발생 시 → Core 스코어에 가산/감점 → 다음 리밸런싱에 반영
#   (별도 satellite 버킷 없음)
#
# T-1: Bull에서 리더 2.5σ 급등 → 해당 섹터 Top Q4 z-score +1.0 가산
#      Neutral에서 리더 2.5σ 급등 → Lag Q1 z-score -1.0 감점
# G-1: 급락 감지 시 → 해당 종목 z-score +1.0 가산
#
# 월말 리밸런싱 시, 직전 hold_d 이내 이벤트를 체크하여 반영.
# =============================================================================

EVENT_PARAMS = {
    "T-1a": {
        "regime": "Bull",
        "surge_sigma": 2.5,
        "hold_d": 7,
        "bonus_z": 1.0,        # Top Q4에 가산
        "tier": "Top_Q4",
    },
    "T-1b": {
        "regime": "Neutral",
        "surge_sigma": 2.5,
        "hold_d": 5,
        "penalty_z": -1.0,     # Lag Q1에 감점
        "tier": "Lag_Q1",
    },
    "G-1": {
        "regimes": ["Bull", "Bear"],
        "hold_d": 20,
        "bonus_z": 1.0,        # 급락 종목에 가산
    },
}

# =============================================================================
# ★ 미확인 상관 쌍 체크리스트
# =============================================================================
#
# 확인 완료:
#   P-5 vs E-5:   신호 -0.005, 수익 -0.37 → 독립 ✅
#   P-5 vs D:     신호 -0.056 → 독립 ✅
#   H vs D-3:     신호 0.22, 수익 0.70, 겹침 10% → 독립 ✅
#   H vs D-1:     신호 0.12, 수익 0.66, 겹침 10% → 독립 ✅
#   T-1 vs H:     신호 -0.004, 겹침 98.5% → 완전독립 ✅
#
# ⚠ 미확인 (Cell 3에서 반드시 확인):
#   G-1 vs D-1    — 방향 반대(급락반전 vs 모멘텀)일 가능성 높지만 미확인
#   G-1 vs P-5    — 이벤트성 vs 구조적 특성, 시간축 다름
#   G-1 vs E-5    — 위와 동일 논리
#   A-3 vs P-5    — accrual vs beta, 논리적 독립이나 미확인
#   A-3 vs E-5    — accrual vs ivol, 논리적 독립이나 미확인
#
# → Cell 3에서 상관 매트릭스 찍고 여기에 결과 업데이트할 것.
# =============================================================================

UNCHECKED_PAIRS = [
    ("G-1", "D-1"),
    ("G-1", "P-5"),
    ("G-1", "E-5"),
    ("A-3", "P-5"),
    ("A-3", "E-5"),
]

# ─── 확인 ─────────────────────────────────────────────────────────────────────
print("="*60)
print("🔨 06_TheForge — Cell 1 로드 완료")
print("="*60)
print(f"  팩터 레지스트리: {len(FACTOR_REGISTRY)}개")
print(f"  레짐: {list(REGIME_FACTOR_MAP.keys())}")
print(f"  미확인 상관 쌍: {len(UNCHECKED_PAIRS)}개")
print(f"  백테스트: {BT_START} ~ {BT_END}")
print(f"  거래비용: {COST_BP}bp 편도")
print()
for regime, mapping in REGIME_FACTOR_MAP.items():
    scores = mapping["scores"]
    events = mapping.get("events", [])
    filters = mapping.get("filters", [])
    avoids = mapping.get("avoids", [])
    print(f"  [{regime}]")
    print(f"    scores:  {scores}")
    if events:  print(f"    events:  {events}")
    if filters: print(f"    filters: {filters}")
    if avoids:  print(f"    avoids:  {avoids}")
    print()
print("→ Cell 2에서 전 팩터 시그널 로드 시작")

🔨 06_TheForge — Cell 1 로드 완료
  팩터 레지스트리: 12개
  레짐: ['Bull', 'Bear', 'Neutral']
  미확인 상관 쌍: 5개
  백테스트: 2013-06 ~ 2026-02
  거래비용: 20bp 편도

  [Bull]
    scores:  ['H', 'G-1_bull', 'P-7']
    events:  ['T-1']

  [Bear]
    scores:  ['H', 'A-3', 'P-5', 'E-5', 'D-1']
    events:  ['G-1_bear']
    filters: ['F-1']
    avoids:  ['G-1b']

  [Neutral]
    scores:  ['H', 'D-3']
    avoids:  ['G-1b']

→ Cell 2에서 전 팩터 시그널 로드 시작


In [2]:
# =============================================================================
# 🔨 06_TheForge.ipynb — Cell 2: 팩터 시그널 로드 + 통합 패널 + 상관행렬
# =============================================================================
#
# [목적]
# 각 05_*.ipynb에서 저장한 parquet 시그널을 로드하고
# 하나의 통합 패널 (date × ticker × 팩터별 z-score)로 병합.
# 미확인 상관 쌍 체크까지 포함.
#
# [로드 파일]
#   h_signal.parquet       → h_z         (H: 섹터 모멘텀)
#   d_signal.parquet       → d1_z, d3_z  (D-1: 모멘텀, D-3: 변동성조정모멘텀)
#   a3_signal.parquet      → a3_z        (A-3: 가치×촉매)
#   p5_e5_signal.parquet   → p5_z, e5_z  (P-5: 저베타, E-5: 저변동)
#   g1_signal.parquet      → g1_bull_z, g1_bear_z, g1b_flag
#   t1_events.parquet      → 이벤트 테이블 (별도 보관)
#   fscore_signal.parquet  → fscore      (F-1: 리스크 필터)
#   p7_signal_panel.parquet→ nsi_score   (P-7: 자사주매입)
#   regime_v4.parquet      → regime, bear_phase
#   ret_1m_wide.parquet    → 월간 수익률 (백테스트용)
#
# [산출물]
#   panel: DataFrame (date, ticker, h_z, d1_z, d3_z, a3_z, p5_z, e5_z,
#                     g1_bull_z, g1_bear_z, g1b_flag, nsi_score, fscore,
#                     regime, bear_phase)
#   ret_1m: wide format 월간 수익률
#   t1_events: 이벤트 테이블
#   corr_matrix: 팩터 간 상관행렬
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

SAVE_DIR = Path(r"C:\QP2\data\interim")

# ─── 1. 팩터 시그널 로드 ──────────────────────────────────────────────────────

print("=" * 60)
print("📦 팩터 시그널 로드 시작")
print("=" * 60)

# H: 섹터 모멘텀
h_sig = pd.read_parquet(SAVE_DIR / "h_signal.parquet")
h_sig["date"] = pd.to_datetime(h_sig["date"])
print(f"  H:    {len(h_sig):>8,} rows, {h_sig['ticker'].nunique()} tickers")

# D: 모멘텀
d_sig = pd.read_parquet(SAVE_DIR / "d_signal.parquet")
d_sig["date"] = pd.to_datetime(d_sig["date"])
print(f"  D:    {len(d_sig):>8,} rows, {d_sig['ticker'].nunique()} tickers")

# A-3: 가치×촉매
a3_sig = pd.read_parquet(SAVE_DIR / "a3_signal.parquet")
a3_sig["date"] = pd.to_datetime(a3_sig["date"])
print(f"  A-3:  {len(a3_sig):>8,} rows, {a3_sig['ticker'].nunique()} tickers")

# P-5 + E-5: 저베타 + 저변동
p5e5_sig = pd.read_parquet(SAVE_DIR / "p5_e5_signal.parquet")
p5e5_sig["date"] = pd.to_datetime(p5e5_sig["date"])
print(f"  P5E5: {len(p5e5_sig):>8,} rows, {p5e5_sig['ticker'].nunique()} tickers")

# G-1: 급락반전 + 급등회피
g1_sig = pd.read_parquet(SAVE_DIR / "g1_signal.parquet")
g1_sig["date"] = pd.to_datetime(g1_sig["date"])
print(f"  G-1:  {len(g1_sig):>8,} rows, {g1_sig['ticker'].nunique()} tickers")

# T-1: 이벤트 테이블 (별도 보관, panel에 안 넣음)
t1_events = pd.read_parquet(SAVE_DIR / "t1_events.parquet")
if len(t1_events) > 0:
    t1_events["date"] = pd.to_datetime(t1_events["date"])
print(f"  T-1:  {len(t1_events):>8,} events")

# F-1: Piotroski F-score
f_sig = pd.read_parquet(SAVE_DIR / "fscore_signal.parquet")
f_sig["date"] = pd.to_datetime(f_sig["date"])
# fscore_signal에서 필요한 컬럼만 추출
f_cols = ["date", "ticker", "fscore"] if "ticker" in f_sig.columns else ["date", "ticker_yahoo", "fscore"]
f_sig = f_sig[f_cols].copy()
if "ticker_yahoo" in f_sig.columns:
    f_sig = f_sig.rename(columns={"ticker_yahoo": "ticker"})
print(f"  F-1:  {len(f_sig):>8,} rows, {f_sig['ticker'].nunique()} tickers")

# P-7: NSI (자사주매입)
p7_sig = pd.read_parquet(SAVE_DIR / "p7_signal_panel.parquet")
p7_sig["date"] = pd.to_datetime(p7_sig["date"])
# nsi_score 컬럼 추출 (= -nsi_z, 자사주매입=높은 점수)
p7_cols = [c for c in ["date", "ticker", "nsi_score"] if c in p7_sig.columns]
if "nsi_score" not in p7_sig.columns:
    # nsi_z가 있으면 부호 반전
    if "nsi_z" in p7_sig.columns:
        p7_sig["nsi_score"] = -p7_sig["nsi_z"]
        p7_cols = ["date", "ticker", "nsi_score"]
p7_sig = p7_sig[p7_cols].copy()
print(f"  P-7:  {len(p7_sig):>8,} rows, {p7_sig['ticker'].nunique()} tickers")

# ─── 2. 레짐 로드 ────────────────────────────────────────────────────────────

regime_v4 = pd.read_parquet(SAVE_DIR / "regime_v4.parquet")
regime_v4.index = pd.to_datetime(regime_v4.index)
regime_v4 = regime_v4.reset_index()
regime_v4 = regime_v4.rename(columns={"index": "date"})
regime_v4["date"] = pd.to_datetime(regime_v4["date"]) + pd.offsets.MonthEnd(0)
regime_map = regime_v4.set_index("date")[["regime", "bear_phase"]].to_dict("index")

print(f"\n  레짐: {regime_v4['regime'].value_counts().to_dict()}")
print(f"  Bear 내부: {regime_v4[regime_v4['regime']=='Bear']['bear_phase'].value_counts().to_dict()}")

# ─── 3. 월간 수익률 로드 ──────────────────────────────────────────────────────

ret_1m = pd.read_parquet(SAVE_DIR / "ret_1m_wide.parquet")
if "date" in ret_1m.columns:
    ret_1m = ret_1m.set_index("date")
ret_1m.index = pd.to_datetime(ret_1m.index)
print(f"\n  ret_1m: {ret_1m.shape[0]} months × {ret_1m.shape[1]} tickers")

# ─── 4. 통합 패널 병합 ───────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("🔗 통합 패널 병합")
print("=" * 60)

# 기준: H 시그널 (전 레짐 항상 ON, 가장 넓은 커버리지)
panel = h_sig[["date", "ticker", "h_z"]].copy()

# D 병합
panel = panel.merge(d_sig[["date", "ticker", "d1_z", "d3_z"]], 
                    on=["date", "ticker"], how="left")

# A-3 병합 (is_cheap 필터 이미 적용됨 → non-cheap은 NaN)
a3_merge = a3_sig[["date", "ticker", "a3_z"]].copy()
panel = panel.merge(a3_merge, on=["date", "ticker"], how="left")

# P-5 + E-5 병합
panel = panel.merge(p5e5_sig[["date", "ticker", "p5_z", "e5_z"]], 
                    on=["date", "ticker"], how="left")

# G-1 병합
panel = panel.merge(g1_sig[["date", "ticker", "g1_bull_z", "g1_bear_z", "g1b_flag"]], 
                    on=["date", "ticker"], how="left")

# P-7 병합
panel = panel.merge(p7_sig, on=["date", "ticker"], how="left")

# F-1 병합
panel = panel.merge(f_sig, on=["date", "ticker"], how="left")

# 레짐 병합
regime_slim = regime_v4[["date", "regime", "bear_phase"]].copy()
panel = panel.merge(regime_slim, on="date", how="left")

# 분석 기간 필터
panel = panel[(panel["date"] >= BT_START) & (panel["date"] <= BT_END)]

print(f"  패널 shape: {panel.shape}")
print(f"  기간: {panel['date'].min()} ~ {panel['date'].max()}")
print(f"  고유 날짜: {panel['date'].nunique()}")
print(f"  고유 종목: {panel['ticker'].nunique()}")

# 팩터별 커버리지
print(f"\n  팩터별 non-null 비율:")
factor_cols = ["h_z", "d1_z", "d3_z", "a3_z", "p5_z", "e5_z", 
               "g1_bull_z", "g1_bear_z", "g1b_flag", "nsi_score", "fscore"]
for col in factor_cols:
    if col in panel.columns:
        pct = panel[col].notna().mean() * 100
        print(f"    {col:14s}: {pct:5.1f}%")

# ─── 5. 상관 매트릭스 (미확인 쌍 포함) ───────────────────────────────────────

print("\n" + "=" * 60)
print("📊 팩터 간 상관행렬 (횡단면 평균)")
print("=" * 60)

# 각 날짜별 횡단면 상관 → 평균
score_cols = ["h_z", "d1_z", "d3_z", "a3_z", "p5_z", "e5_z", 
              "g1_bull_z", "g1_bear_z", "nsi_score"]

corr_by_date = []
for dt, grp in panel.groupby("date"):
    sub = grp[score_cols].dropna(how="all")
    if len(sub) > 30:  # 최소 30종목
        corr_by_date.append(sub.corr())

if corr_by_date:
    corr_matrix = pd.concat(corr_by_date).groupby(level=0).mean()
    
    # 소수점 3자리로 표시
    print(corr_matrix.round(3).to_string())
    
    # 미확인 쌍 집중 확인
    print("\n" + "-" * 50)
    print("미확인 쌍 상관계수:")
    print("-" * 50)
    check_pairs = [
        ("g1_bull_z", "d1_z",    "G-1 vs D-1"),
        ("g1_bear_z", "d1_z",    "G-1(Bear) vs D-1"),
        ("g1_bull_z", "p5_z",    "G-1 vs P-5"),
        ("g1_bear_z", "e5_z",    "G-1(Bear) vs E-5"),
        ("a3_z",      "p5_z",    "A-3 vs P-5"),
        ("a3_z",      "e5_z",    "A-3 vs E-5"),
        ("p5_z",      "e5_z",    "P-5 vs E-5 (확인용)"),
        ("h_z",       "d3_z",    "H vs D-3 (확인용)"),
    ]
    for c1, c2, label in check_pairs:
        if c1 in corr_matrix.columns and c2 in corr_matrix.index:
            val = corr_matrix.loc[c1, c2]
            flag = "⚠" if abs(val) > 0.5 else "✅"
            print(f"  {flag} {label:25s}: {val:+.3f}")
        else:
            print(f"  ❓ {label:25s}: 데이터 없음")
else:
    print("  ⚠ 상관행렬 계산 불가 (데이터 부족)")
    corr_matrix = pd.DataFrame()

# ─── 6. 레짐별 팩터 분포 요약 ─────────────────────────────────────────────────

print("\n" + "=" * 60)
print("📈 레짐별 데이터 분포")
print("=" * 60)

for regime in ["Bull", "Bear", "Neutral"]:
    sub = panel[panel["regime"] == regime]
    n_months = sub["date"].nunique()
    n_rows = len(sub)
    print(f"\n  [{regime}] {n_months} months, {n_rows:,} rows")
    
    # 해당 레짐에서 사용할 팩터 커버리지
    regime_factors = REGIME_FACTOR_MAP[regime]["scores"]
    for factor_name in regime_factors:
        # 팩터명 → 컬럼명 매핑
        col_map = {
            "H": "h_z", "D-3": "d3_z", "D-1": "d1_z", "A-3": "a3_z",
            "P-5": "p5_z", "E-5": "e5_z", "G-1_bull": "g1_bull_z",
            "G-1_bear": "g1_bear_z", "P-7": "nsi_score",
        }
        col = col_map.get(factor_name, "")
        if col and col in sub.columns:
            pct = sub[col].notna().mean() * 100
            print(f"    {factor_name:10s} ({col:12s}): {pct:5.1f}% non-null")

print("\n" + "=" * 60)
print("✅ Cell 2 완료: 통합 패널 + 상관행렬")
print(f"   panel: {panel.shape}")
print(f"   t1_events: {len(t1_events)} events")
print(f"   ret_1m: {ret_1m.shape}")
print("→ Cell 3에서 레짐별 동일가중 백테스트 시작")
print("=" * 60)

📦 팩터 시그널 로드 시작
  H:     205,224 rows, 503 tickers
  D:      69,834 rows, 467 tickers
  A-3:    19,426 rows, 492 tickers
  P5E5:  206,504 rows, 503 tickers
  G-1:    72,374 rows, 503 tickers
  T-1:     1,533 events
  F-1:    88,324 rows, 503 tickers
  P-7:   387,310 rows, 503 tickers

  레짐: {'Bull': 551, 'Bear': 170, 'Neutral': 44}
  Bear 내부: {'declining': 151, 'recovering': 19}

  ret_1m: 770 months × 277 tickers

🔗 통합 패널 병합
  패널 shape: (76456, 15)
  기간: 2013-06-30 00:00:00 ~ 2026-01-31 00:00:00
  고유 날짜: 152
  고유 종목: 503

  팩터별 non-null 비율:
    h_z           : 100.0%
    d1_z          :  85.2%
    d3_z          :  90.7%
    a3_z          :  10.6%
    p5_z          :  95.3%
    e5_z          :  95.8%
    g1_bull_z     :  94.0%
    g1_bear_z     :  92.7%
    g1b_flag      :  94.0%
    nsi_score     :  84.1%
    fscore        :  93.5%

📊 팩터 간 상관행렬 (횡단면 평균)
             h_z   d1_z   d3_z   a3_z   p5_z   e5_z  g1_bull_z  g1_bear_z  nsi_score
a3_z      -0.032  0.080  0.031  1.000 -0.035  0.0